In [2]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os


# Stop existing SparkContext so new settings apply

if SparkContext._active_spark_context is not None:
    SparkContext._active_spark_context.stop()


# Set Java home for WSL

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


# Configure Spark with PostgreSQL driver

conf = SparkConf() \
    .setAppName("CSV_to_Postgres_ETL") \
    .setMaster("local[*]") \
    .set("spark.jars", "/home/lana/ETL_Pipeline_4/jars/postgresql-42.7.3.jar")

# Create Spark session

sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession.builder.config(conf=sc.getConf()).getOrCreate()

print("Spark session created successfully!")
print("Spark version:", spark.version)


# EXTRACT: Read data from CSV file

csv_path = "/home/lana/ETL_Pipeline_1/Softwork.csv"


 # change this to your file path

df = spark.read.csv(csv_path, header=True, inferSchema=True)
print("Data extracted from CSV:")
df.show(5)


# TRANSFORM: 

df_transformed = df.filter(col("age") > 25)


print("Data transformed:")
df_transformed.show(5)


# LOAD: Write to PostgreSQL

pg_url = "jdbc:postgresql://localhost:5432/postgres"
pg_properties = {
    "user": "postgres",
    "password": "lana",         
    "driver": "org.postgresql.Driver"
}

df_transformed.write.jdbc(
    url=pg_url,
    table="employee_py",
    mode="overwrite",
    properties=pg_properties
)

print("Data successfully written to PostgreSQL table: employee_py")



Spark session created successfully!
Spark version: 4.0.1
Data extracted from CSV:
+-----------+-----------------+---------+----------+------+-------------------+---------------+---+--------------------+-----------------+----------+------------------+
|employee_id|       department|   region| education|gender|recruitment_channel|no_of_trainings|age|previous_year_rating|length_of_service|awards_won|avg_training_score|
+-----------+-----------------+---------+----------+------+-------------------+---------------+---+--------------------+-----------------+----------+------------------+
|       8724|       Technology|region_26|Bachelor's|     m|           sourcing|              1| 24|                NULL|                1|         0|                77|
|      74430|               HR| region_4|Bachelor's|     f|              other|              1| 31|                   3|                5|         0|                51|
|      72255|Sales & Marketing|region_13|Bachelor's|     m|              

Data successfully written to PostgreSQL table: employee_py
